Sustituye 1234 por los últimos 4 dígitos de tu matrícula. Ejecuta esta celda primero para generar tu corpus de trabajo individual:

In [1]:
import random

# INSTRUCCIÓN: Sustituye 1234 por los últimos 4 dígitos de tu matrícula
MATRICULA = 250

# Parámetros individuales autogenerados (NO MODIFICAR)
NUM_TOPICOS_LDA = 3 + (MATRICULA % 3)  # Será 3, 4 o 5 tópicos
SEMILLA = MATRICULA

print(f"Configuración individual -> Tópicos LDA (K): {NUM_TOPICOS_LDA} | Semilla: {SEMILLA}")


Configuración individual -> Tópicos LDA (K): 4 | Semilla: 250


In [ ]:
Copia y ejecuta exactamente este corpus de noticias corporativas y reportes técnicos:

In [2]:
corpus_documentos = [
    "El 14/03/2025, Grupo Alfa (ID: TX-9082-MX) adquirió el 60% de Solaris Tech por $45.5M USD en Guadalajara.",
    "La empresa Beta Corp (ID: TX-1104-US) NO adquirió a CloudNet; la operación por $120M USD fue cancelada el 2025-04-01.",
    "El Dr. Roberto Méndez, director de BioSalud, anunció una inversión de €18.2M EUR para investigación genómica en Madrid.",
    "Falso reporte: el código REF-9999-MX no corresponde a una transacción válida, aunque mencione a TecNova y $10M USD.",
    "Kinetix Robotics concretó la compra de AeroDrones (ID: TX-4421-ES) por un monto de $8.75M USD el 19/05/2025.",
    "Los servidores con kernel Linux presentaron fallos de latencia en el clúster de Kubernetes tras actualizar el firewall.",
    "La vulnerabilidad crítica en el protocolo SSH permitió escalada de privilegios en nodos Ubuntu sin parchear.",
    "El nuevo algoritmo de cifrado post-cuántico reduce el consumo de memoria RAM en microcontroladores ARM.",
    "La cosecha de agave azul en Jalisco enfrentó sequías severas, elevando el costo por tonelada de materia prima.",
    "Nuevos fertilizantes orgánicos mejoran la retención de humedad en cultivos de maíz y aguacate en el occidente.",
    "El sistema de riego automatizado con sensores IoT redujo un 30% el desperdicio de agua en invernaderos."
]

# Reordenar corpus según matrícula
random.seed(SEMILLA)
random.shuffle(corpus_documentos)


In [3]:
import re
import pandas as pd

# (?P<nombre>...) = grupo nombrado
patron = re.compile(
    r"ID:\s*(?P<id_tx>TX-\d{4}-[A-Z]{2}).*?"
    r"(?P<comprador>[A-Z][\w\s\.]+?)\s+(?:adquirió|concretó la compra de|adquirió a)\s+"
    r"(?P<objetivo>[A-Z][\w\s]+?)\s+.*?por\s+(?:un monto de\s+)?"
    r"(?P<monto>[\$€]\d+(?:\.\d+)?M\s*(?:USD|EUR))"
    , re.IGNORECASE | re.DOTALL
)

# Patrones trampa para descartar
patron_falso = re.compile(r"REF-\d{4}-[A-Z]{2}|Falso reporte|no corresponde", re.IGNORECASE)
patron_cancelado = re.compile(r"NO adquirió|cancelada|cancelado|no se haya concretado", re.IGNORECASE)

relaciones = []

for texto in corpus_documentos:
    # 1. Ignorar falsos y cancelados - sin borrar a mano
    if patron_falso.search(texto):
        continue
    if patron_cancelado.search(texto):
        continue

    m = patron.search(texto)
    if m:
        relaciones.append({
            'id_tx': m.group('id_tx'),
            'comprador': m.group('comprador').strip(),
            'objetivo': m.group('objetivo').strip(),
            'monto': m.group('monto').strip()
        })

df_relaciones = pd.DataFrame(relaciones)
print(df_relaciones)

Empty DataFrame
Columns: []
Index: []


In [5]:
import spacy
from spacy.pipeline import EntityRuler

# Descargar automáticamente el modelo si no está instalado
try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    import spacy.cli
    spacy.cli.download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")

# Agrega EntityRuler ANTES de ner para que no sobreescriba
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Patrones de MONEY - de la Parte 1
patrones_money = [
    {"label": "MONEY", "pattern": [{"TEXT": {"REGEX": r"[\$€]\d+(\.\d+)?M"}}, {"TEXT": {"REGEX": r"USD|EUR"}}]},
    {"label": "MONEY", "pattern": [{"TEXT": {"REGEX": r"[\$€]\d+(\.\d+)?"}}, {"TEXT": {"REGEX": r"M"}}, {"TEXT": {"REGEX": r"USD|EUR"}}]},
]
ruler.add_patterns(patrones_money)

# Probar en los primeros 4 documentos de tu lista ya mezclada
for doc_text in corpus_documentos[:4]:
    doc = nlp(doc_text)
    print("Texto:", doc_text[:80], "...")
    for ent in doc.ents:
        print(f"  {ent.text} -> {ent.label_}")
    print()

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Texto: La cosecha de agave azul en Jalisco enfrentó sequías severas, elevando el costo  ...
  Jalisco -> LOC

Texto: Falso reporte: el código REF-9999-MX no corresponde a una transacción válida, au ...
  TecNova -> MISC

Texto: Nuevos fertilizantes orgánicos mejoran la retención de humedad en cultivos de ma ...
  maíz -> LOC

Texto: El nuevo algoritmo de cifrado post-cuántico reduce el consumo de memoria RAM en  ...
  RAM -> MISC
  ARM -> MISC



In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Preprocesamiento obligatorio: solo lemas NOUN y ADJ
def limpiar(texto):
    doc = nlp(texto)
    lemas = [token.lemma_.lower() for token in doc
             if not token.is_stop and not token.is_punct
             and token.pos_ in ["NOUN", "ADJ"]]
    return " ".join(lemas)

corpus_limpio = [limpiar(t) for t in corpus_documentos]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus_limpio)

lda = LatentDirichletAllocation(
    n_components=NUM_TOPICOS_LDA,  # 4 en tu caso
    random_state=SEMILLA
)
lda.fit(X)

palabras = vectorizer.get_feature_names_out()
for i, topico in enumerate(lda.components_):
    top_5 = [palabras[j] for j in topico.argsort()[-5:][::-1]]
    print(f"Tópico {i}: {', '.join(top_5)}")

Tópico 0: tx, tonelada, sequía, severo, materia
Tópico 1: sistema, sensor, riego, invernadero, automatizado
Tópico 2: vulnerabilidad, ubuntu, protocolo, privilegio, nodo
Tópico 3: tx, servidor, latencia, firewall, fallo


# Por qué tuviste que usar EntityRuler para la etiqueta MONEY en es_core_news_sm mientras que en modelos en inglés (en_core_web_sm) sí existe por defecto?

En es_core_news_sm no existe la etiqueta MONEY porque ese modelo se entrenó con AnCora / WikiNER, que solo tiene PER, ORG, LOC y MISC. En inglés, en_core_web_sm sí tiene MONEY porque se entrenó con OntoNotes 5, que sí incluye dinero, fechas, cantidades, etc. Por eso tuvimos que usar EntityRuler para crear MONEY con reglas.